# Week 1: Data Understanding & EDA

## Goals:
- Know your datasets inside out
- Organize all datasets (Footwear, Instacart)
- Explore each dataset: sizes, columns, missing values, unique IDs
- Do EDA (exploratory data analysis)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load and Explore Footwear Manufacturing Dataset

In [ ]:
# Load datasets
customer_orders = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Customer_Order.csv', sep=';')
picking_waves = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Picking_Wave.csv', sep=';')
products = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Product.csv', sep=';')

print("Customer Orders Shape:", customer_orders.shape)
print("Picking Waves Shape:", picking_waves.shape)
print("Products Shape:", products.shape)

In [ ]:
# Display first few rows of each dataset
print("Customer Orders:")
display(customer_orders.head())

print("\nPicking Waves:")
display(picking_waves.head())

print("\nProducts:")
display(products.head())

In [ ]:
# Check data types and missing values
print("Customer Orders Info:")
print(customer_orders.info())
print("\nMissing values in Customer Orders:")
print(customer_orders.isnull().sum())

In [ ]:
print("Picking Waves Info:")
print(picking_waves.info())
print("\nMissing values in Picking Waves:")
print(picking_waves.isnull().sum())

In [ ]:
print("Products Info:")
print(products.info())
print("\nMissing values in Products:")
print(products.isnull().sum())

## 2. Load and Explore Instacart Dataset

In [ ]:
# Load Instacart datasets (sampling for performance)
aisles = pd.read_csv('../../Instacart_dataset/aisles.csv')
departments = pd.read_csv('../../Instacart_dataset/departments.csv')
products_instacart = pd.read_csv('../../Instacart_dataset/products.csv')
orders = pd.read_csv('../../Instacart_dataset/orders.csv')
order_products_prior = pd.read_csv('../../Instacart_dataset/order_products__prior.csv')
order_products_train = pd.read_csv('../../Instacart_dataset/order_products__train.csv')

print("Aisles Shape:", aisles.shape)
print("Departments Shape:", departments.shape)
print("Products Instacart Shape:", products_instacart.shape)
print("Orders Shape:", orders.shape)
print("Order Products Prior Shape:", order_products_prior.shape)
print("Order Products Train Shape:", order_products_train.shape)

In [ ]:
# Display first few rows
print("Aisles:")
display(aisles.head())

print("\nDepartments:")
display(departments.head())

print("\nProducts Instacart:")
display(products_instacart.head())

In [ ]:
print("Orders (first 5 rows):")
display(orders.head())

print("\nOrder Products Prior (first 5 rows):")
display(order_products_prior.head())

## 3. EDA for Footwear Dataset

### Order Size Distribution (lines/order)

In [ ]:
# Order size distribution
order_sizes = customer_orders.groupby('orderNumber').size()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(order_sizes, bins=50, edgecolor='black')
plt.title('Distribution of Order Sizes (Lines per Order)')
plt.xlabel('Number of Lines per Order')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.boxplot(order_sizes)
plt.title('Boxplot of Order Sizes')
plt.ylabel('Number of Lines per Order')

plt.tight_layout()
plt.show()

print(f"Order Size Statistics:\n{order_sizes.describe()}")

### SKU Popularity (ABC Curve)

In [ ]:
# SKU popularity analysis
sku_popularity = customer_orders['Reference'].value_counts()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(sku_popularity, bins=50, edgecolor='black')
plt.title('Distribution of SKU Popularity')
plt.xlabel('Number of Orders per SKU')
plt.ylabel('Frequency')
plt.yscale('log')

plt.subplot(1, 2, 2)
# ABC analysis - cumulative percentage
sku_sorted = sku_popularity.sort_values(ascending=False)
cumulative_percentage = sku_sorted.cumsum() / sku_sorted.sum() * 100
sku_index = range(1, len(cumulative_percentage) + 1)

plt.plot(sku_index, cumulative_percentage)
plt.title('ABC Analysis - Cumulative Percentage of Orders')
plt.xlabel('SKU Rank')
plt.ylabel('Cumulative Percentage of Orders (%)')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Top 10 Most Popular SKUs:\n{sku_popularity.head(10)}")

### Pick-to-Pick Intervals

In [ ]:
# Analyze pick-to-pick intervals by operator
# Convert creationDate to datetime
customer_orders['creationDate'] = pd.to_datetime(customer_orders['creationDate'], format='%d/%m/%Y %H:%M')

# Sort by operator and creation date
customer_orders_sorted = customer_orders.sort_values(['operator', 'creationDate'])

# Calculate time differences for each operator
customer_orders_sorted['time_diff'] = customer_orders_sorted.groupby('operator')['creationDate'].diff()
customer_orders_sorted['time_diff_seconds'] = customer_orders_sorted['time_diff'].dt.total_seconds()

# Remove NaN values
time_intervals = customer_orders_sorted.dropna(subset=['time_diff_seconds'])

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(time_intervals['time_diff_seconds'].dropna(), bins=100, edgecolor='black')
plt.title('Distribution of Pick-to-Pick Time Intervals')
plt.xlabel('Time Interval (seconds)')
plt.ylabel('Frequency')
plt.yscale('log')

plt.subplot(1, 2, 2)
sns.boxplot(data=time_intervals, x='operator', y='time_diff_seconds')
plt.title('Pick-to-Pick Time Intervals by Operator')
plt.xlabel('Operator')
plt.ylabel('Time Interval (seconds)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"Time Interval Statistics (seconds):\n{time_intervals['time_diff_seconds'].describe()}")

## 4. EDA for Instacart Dataset

### Order Size Distribution

In [ ]:
# Sample a subset of order_products_prior for performance
order_products_sample = order_products_prior.sample(n=100000, random_state=42)

# Order size distribution
instacart_order_sizes = order_products_sample.groupby('order_id').size()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(instacart_order_sizes, bins=50, edgecolor='black')
plt.title('Distribution of Order Sizes (Instacart)')
plt.xlabel('Number of Products per Order')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.boxplot(instacart_order_sizes)
plt.title('Boxplot of Order Sizes (Instacart)')
plt.ylabel('Number of Products per Order')

plt.tight_layout()
plt.show()

print(f"Instacart Order Size Statistics:\n{instacart_order_sizes.describe()}")

### Product Popularity (ABC Analysis)

In [ ]:
# Product popularity
product_popularity = order_products_sample['product_id'].value_counts()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(product_popularity, bins=50, edgecolor='black')
plt.title('Distribution of Product Popularity (Instacart)')
plt.xlabel('Number of Orders per Product')
plt.ylabel('Frequency')
plt.yscale('log')

plt.subplot(1, 2, 2)
# ABC analysis
product_sorted = product_popularity.sort_values(ascending=False)
cumulative_percentage = product_sorted.cumsum() / product_sorted.sum() * 100
product_index = range(1, len(cumulative_percentage) + 1)

plt.plot(product_index, cumulative_percentage)
plt.title('ABC Analysis - Cumulative Percentage of Orders (Instacart)')
plt.xlabel('Product Rank')
plt.ylabel('Cumulative Percentage of Orders (%)')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Top 10 Most Popular Products:\n{product_popularity.head(10)}")

## 5. Dataset Summaries

### Footwear Manufacturing Dataset Summary:
1. Contains customer orders with details like order number, product reference, size, quantity, creation date, wave number, and operator.
2. Picking wave data includes wave number, product reference, size, quantity to pick, locations, and operator.
3. Product data includes reference, ABC classification code, and sector.

### Instacart Dataset Summary:
1. Comprehensive grocery dataset with aisles, departments, products, orders, and order products.
2. Detailed information on customer orders including order time, day of week, and reorder information.
3. Product hierarchy with aisle and department categorization.